# Speculative Decoding Experiment

**Phases 1-8**: Data preparation -> Baseline -> AR Grid A (0.5B) -> AR Grid B (1.5B) -> Quality/Error Analysis -> Synthesis/Visualization -> ACSD runs -> ACSD evaluation summary

| Config | Draft | k | Regime |
|--------|-------|---|--------|
| Baseline x 2 | - | - | det / stoch |
| Grid A x 6 | Qwen2.5-0.5B | {4,8,16} | det / stoch |
| Grid B x 6 | Qwen2.5-1.5B | {4,8,16} | det / stoch |
| ACSD x N | 0.5B -> 1.5B cascade | adaptive {4,8,16} | det / stoch |

**Total runs:** 2 baselines + 12 AR speculative + ACSD configurations

In [1]:
import os
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

# Ensure src/ is on the path (local execution).
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Force offline-only model usage from local cache.
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Core imports
import torch
import pandas as pd
from tqdm.auto import tqdm

import baseline as baseline_module
import speculative as speculative_module

from config import (
    TARGET_MODEL_ID, DRAFT_MODELS, DATASETS, REGIMES,
    DRAFT_LENGTHS, RESULTS_DIR, STABILITY_DIR, FIGURES_DIR,
    SEED, STABILITY_SEEDS, MANIFESTS_DIR,
)
from utils import set_seed, get_env_info, write_csv

# Print environment
env = get_env_info()
for k, v in env.items():
    print(f"{k}: {v}")
print("Hugging Face mode: offline-only (local cache)")


def install_notebook_progress_helpers():
    helper_version = 2
    if globals().get("_NOTEBOOK_PROGRESS_INSTALLED_VERSION") == helper_version:
        return

    def _short_regime(regime_name: str) -> str:
        return "det" if regime_name == "deterministic" else "stoch"

    def _eta_clock(elapsed_s: float, done_n: int, total_n: int) -> tuple[float, str]:
        if done_n <= 0 or total_n <= done_n:
            return 0.0, "--:--:--"
        remaining_s = max((elapsed_s / done_n) * (total_n - done_n), 0.0)
        eta_ts = datetime.now() + timedelta(seconds=remaining_s)
        return remaining_s, eta_ts.strftime("%H:%M:%S")

    def notebook_run_baseline(
        data: dict[str, list[dict]],
        regime_name: str,
        model=None,
        tokenizer=None,
    ) -> list[dict]:
        regime = REGIMES[regime_name]
        if model is None or tokenizer is None:
            model, tokenizer = baseline_module.load_target_model()

        results = []
        total = sum(len(v) for v in data.values())
        done = 0
        label = f"base {_short_regime(regime_name)}"
        bar = tqdm(
            total=total,
            desc=label,
            dynamic_ncols=True,
            leave=True,
            mininterval=0.5,
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )
        started_at = time.perf_counter()

        try:
            for task_name, samples in data.items():
                max_new_tokens = DATASETS[task_name]["max_new_tokens"]
                task_total = len(samples)
                for task_idx, sample in enumerate(samples, start=1):
                    bar.set_postfix_str(f"task={task_name} {task_idx}/{task_total}")
                    set_seed(SEED)

                    sample_started_at = time.perf_counter()
                    out = baseline_module.run_baseline_sample(
                        model,
                        tokenizer,
                        sample["prompt"],
                        max_new_tokens,
                        regime.temperature,
                        regime.top_p,
                    )
                    sample_elapsed = time.perf_counter() - sample_started_at

                    row = {
                        "sample_id": sample["sample_id"],
                        "task": task_name,
                        "regime": regime_name,
                        **out,
                    }
                    results.append(row)

                    done += 1
                    elapsed = time.perf_counter() - started_at
                    avg_sample_s = elapsed / done if done else 0.0
                    remaining_s, eta_clock = _eta_clock(elapsed, done, total)
                    bar.update(1)
                    bar.set_postfix_str(
                        f"task={task_name} {task_idx}/{task_total} | last={sample_elapsed:.1f}s | avg={avg_sample_s:.1f}s | rem={remaining_s/60:.1f}m | eta={eta_clock}"
                    )
        finally:
            bar.close()

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        csv_path = RESULTS_DIR / f"baseline_{regime_name}.csv"
        write_csv(csv_path, results)
        print(f"Saved -> {csv_path}")
        return results

    def notebook_run_speculative_grid(
        data: dict[str, list[dict]],
        draft_label: str,
        k: int,
        regime_name: str,
        target_model=None,
        target_tokenizer=None,
        draft_model=None,
        draft_tokenizer=None,
        seed: int = SEED,
    ) -> list[dict]:
        regime = REGIMES[regime_name]
        if target_model is None:
            target_model, target_tokenizer = baseline_module.load_target_model()
        if draft_model is None:
            draft_model, draft_tokenizer = speculative_module.load_draft_model(draft_label)

        results = []
        total = sum(len(v) for v in data.values())
        done = 0
        regime_short = _short_regime(regime_name)
        label = f"spec {draft_label} k={k} {regime_short}"
        bar = tqdm(
            total=total,
            desc=label,
            dynamic_ncols=True,
            leave=True,
            mininterval=0.5,
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
        )
        started_at = time.perf_counter()

        try:
            for task_name, samples in data.items():
                max_new_tokens = DATASETS[task_name]["max_new_tokens"]
                task_total = len(samples)
                for task_idx, sample in enumerate(samples, start=1):
                    bar.set_postfix_str(f"task={task_name} {task_idx}/{task_total}")
                    set_seed(seed)

                    sample_started_at = time.perf_counter()
                    out = speculative_module.speculative_decode_sample(
                        target_model,
                        draft_model,
                        target_tokenizer,
                        sample["prompt"],
                        max_new_tokens,
                        k,
                        regime.temperature,
                        regime.top_p,
                    )
                    sample_elapsed = time.perf_counter() - sample_started_at

                    row = {
                        "sample_id": sample["sample_id"],
                        "task": task_name,
                        "draft": draft_label,
                        "k": k,
                        "regime": regime_name,
                        "seed": seed,
                        **{key: val for key, val in out.items() if key != "verify_log"},
                    }
                    results.append(row)

                    done += 1
                    elapsed = time.perf_counter() - started_at
                    avg_sample_s = elapsed / done if done else 0.0
                    remaining_s, eta_clock = _eta_clock(elapsed, done, total)
                    bar.update(1)
                    bar.set_postfix_str(
                        f"task={task_name} {task_idx}/{task_total} | last={sample_elapsed:.1f}s | avg={avg_sample_s:.1f}s | rem={remaining_s/60:.1f}m | eta={eta_clock}"
                    )
        finally:
            bar.close()

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        csv_path = RESULTS_DIR / f"spec_{draft_label}_k{k}_{regime_short}.csv"
        write_csv(csv_path, results)
        print(f"Saved -> {csv_path}")
        return results

    globals()["notebook_run_baseline"] = notebook_run_baseline
    globals()["notebook_run_speculative_grid"] = notebook_run_speculative_grid
    globals()["run_baseline"] = notebook_run_baseline
    globals()["run_speculative_grid"] = notebook_run_speculative_grid

    baseline_module.run_baseline = notebook_run_baseline
    speculative_module.run_speculative_grid = notebook_run_speculative_grid

    globals()["_NOTEBOOK_PROGRESS_INSTALLED"] = True
    globals()["_NOTEBOOK_PROGRESS_INSTALLED_VERSION"] = helper_version
    print("Notebook progress helpers installed")


install_notebook_progress_helpers()



python: 
torch: 2.11.0+cu128
cuda_available: True
cuda_version: 12.8
gpu_name: NVIDIA GeForce RTX 4090
device: cuda
transformers: 5.5.4
Hugging Face mode: offline-only (local cache)
Notebook progress helpers installed


## Phase 1 — Data Preparation & Reproducibility Lock

Load GSM8K (300), MMLU (5×100), CNN/DailyMail (200) with `seed=42`, freeze manifests.

In [ ]:
# Hugging Face offline-only mode (force local cache usage).
import os

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("Offline-only mode enabled: local Hugging Face cache only.")


In [ ]:
from data_loader import load_all_datasets, freeze_manifests, save_full_data, load_from_manifests

# Check if manifests already exist (skip download if so)
manifests_exist = all(
    (MANIFESTS_DIR / f"{t}_data.json").exists()
    for t in ["gsm8k", "mmlu", "cnndm"]
)

if manifests_exist:
    print("Manifests found — loading from disk (no re-download)")
    data = load_from_manifests()
else:
    print("Downloading and sampling datasets…")
    data = load_all_datasets()
    freeze_manifests(data)
    save_full_data(data)

# Quick sanity check
for task, samples in data.items():
    print(f"  {task}: {len(samples)} samples, first id: {samples[0]['sample_id']}")

### Verify Tokenizer Compatibility

In [ ]:
from data_loader import verify_tokenizer_compatibility
import os

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

compatible = verify_tokenizer_compatibility()
assert compatible, "Tokenizer mismatch; cannot proceed with speculative decoding."


## Phase 2 — Baseline Benchmarking

Run Qwen2.5-3B-Instruct autoregressive decoding on all 1,000 samples in both regimes.

In [ ]:
import os
import sys
import importlib
from pathlib import Path

# Ensure src/ is on the path (local execution).
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

cwd = Path.cwd().resolve()
def _find_project_root(start):
    for c in [start, *start.parents]:
        if (c / "src" / "baseline.py").exists():
            return c
    return start

project_root = _find_project_root(cwd)
SRC_DIR = str(project_root / "src")
if not (project_root / "src" / "baseline.py").exists():
    raise ModuleNotFoundError(
        f"Could not find src/baseline.py from cwd={cwd}."
    )
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Force-refresh modules to avoid stale target-model config in long-lived kernels.
import config as config_module
import baseline as baseline_module
import runtime as runtime_module
config_module = importlib.reload(config_module)
baseline_module = importlib.reload(baseline_module)
runtime_module = importlib.reload(runtime_module)

from runtime import bootstrap_notebook, ensure_data, ensure_target_model
from baseline import run_baseline
from config import TARGET_MODEL_ID

bootstrap_notebook()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

data = ensure_data(globals())
target_model, target_tokenizer = ensure_target_model(globals())

loaded_target = str(getattr(target_model, "name_or_path", ""))
print(f"Configured target model: {TARGET_MODEL_ID}")
print(f"Loaded target model: {loaded_target}")
assert TARGET_MODEL_ID.endswith("Qwen2.5-3B-Instruct"), f"Unexpected TARGET_MODEL_ID: {TARGET_MODEL_ID}"
assert loaded_target == TARGET_MODEL_ID, f"Target mismatch: loaded={loaded_target}, expected={TARGET_MODEL_ID}"

if "baseline_det" not in globals() and (RESULTS_DIR / "baseline_deterministic.csv").exists():
    baseline_det = pd.read_csv(RESULTS_DIR / "baseline_deterministic.csv").to_dict(orient="records")
if "baseline_stoch" not in globals() and (RESULTS_DIR / "baseline_stochastic.csv").exists():
    baseline_stoch = pd.read_csv(RESULTS_DIR / "baseline_stochastic.csv").to_dict(orient="records")


In [ ]:
# --- Deterministic baseline ---
print("=" * 60)
print("Baseline: DETERMINISTIC regime")
print("=" * 60)
if globals().get("baseline_det"):
    print("Using existing deterministic baseline from memory/disk")
else:
    baseline_det = notebook_run_baseline(data, "deterministic", target_model, target_tokenizer)



In [ ]:
# --- Stochastic baseline ---
print("=" * 60)
print("Baseline: STOCHASTIC regime")
print("=" * 60)
if globals().get("baseline_stoch"):
    print("Using existing stochastic baseline from memory/disk")
else:
    baseline_stoch = notebook_run_baseline(data, "stochastic", target_model, target_tokenizer)



In [ ]:
# --- Evaluate baseline quality ---
from evaluate import evaluate_results

print("\nBaseline quality — Deterministic:")
base_quality_det = evaluate_results(baseline_det, data)

print("\nBaseline quality — Stochastic:")
base_quality_stoch = evaluate_results(baseline_stoch, data)

In [ ]:
# --- Quick latency summary ---
from metrics import compute_latency_metrics

print("\nBaseline latency — Deterministic:")
base_lat_det = compute_latency_metrics(baseline_det)
for k, v in base_lat_det.items():
    print(f"  {k}: {v}")

print("\nBaseline latency — Stochastic:")
base_lat_stoch = compute_latency_metrics(baseline_stoch)
for k, v in base_lat_stoch.items():
    print(f"  {k}: {v}")

## Phase 2.5 — Dual-GPU 3B/3B Speculative Pilot (First 200 Samples)

Load target 3B on `cuda:0` and a second 3B as draft on `cuda:1`, then run speculative decoding on the first 200 samples (`k=4`) for both regimes. Output CSV format matches existing speculative result files for unified analysis.

In [ ]:
import os
import sys
from pathlib import Path

import torch

# Ensure src/ is importable even when this cell is run independently.
os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src" / "runtime.py").exists()), cwd)
SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from runtime import bootstrap_notebook, ensure_data, ensure_dual_3b_results
from config import TARGET_MODEL_ID, RESULTS_DIR

bootstrap_notebook()

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Dual-GPU run requires at least 2 CUDA devices: cuda:0 and cuda:1")

data = ensure_data(globals())

# Show exactly how many rows are taken per task from the global first-200 slice.
remaining = 200
subset_breakdown = {}
for task_name, samples in data.items():
    if remaining <= 0:
        break
    take_n = min(len(samples), remaining)
    if take_n > 0:
        subset_breakdown[task_name] = take_n
        remaining -= take_n

print(f"Target model id: {TARGET_MODEL_ID}")
print(f"CUDA devices: {torch.cuda.device_count()}")
print(f"Subset breakdown (first 200 total): {subset_breakdown}")
print("Live compare: showing two progress bars (draft on cuda:1, target verify on cuda:0) with separate ETA")

spec_results_3b_dual = ensure_dual_3b_results(
    globals(),
    k=4,
    max_samples=200,
    target_device="cuda:0",
    draft_device="cuda:1",
    regimes=("deterministic", "stochastic"),
    draft_label="3B_dual",
    show_realtime_progress=True,
    force_rerun=True,
 )

print("Saved CSV files:")
for regime_name in ("deterministic", "stochastic"):
    short = "det" if regime_name == "deterministic" else "stoch"
    print(f"  - {RESULTS_DIR / f'spec_3B_dual_k4_{short}.csv'}")

print(f"Completed dual-3B configs: {list(spec_results_3b_dual.keys())}")

## Phase 3 — Speculative Grid A: Qwen2.5-0.5B Draft

Run speculative decoding with 0.5B draft across k ∈ {4, 8, 16} × {deterministic, stochastic} = 6 configs.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from runtime import bootstrap_notebook, ensure_data, ensure_target_model, ensure_baseline_results, ensure_draft_model, ensure_spec_results
from speculative import run_speculative_grid
from config import TARGET_MODEL_ID

bootstrap_notebook()
if "install_notebook_progress_helpers" in globals():
    install_notebook_progress_helpers()

data = ensure_data(globals())
ensure_baseline_results(globals())
spec_results_05 = ensure_spec_results(globals(), "0.5B")
target_model, target_tokenizer = ensure_target_model(globals())
draft_05_model, draft_05_tokenizer = ensure_draft_model(globals(), "0.5B")

loaded_target = str(getattr(target_model, "name_or_path", ""))
print(f"Configured target model: {TARGET_MODEL_ID}")
print(f"Loaded target model: {loaded_target}")
assert loaded_target == TARGET_MODEL_ID, f"Target mismatch: loaded={loaded_target}, expected={TARGET_MODEL_ID}"


In [ ]:
from config import DRAFT_LENGTHS, REGIMES

spec_results_05 = globals().get("spec_results_05", {})
spec_grid_runner = globals().get("notebook_run_speculative_grid")
if spec_grid_runner is None:
    raise RuntimeError("Notebook progress runner is not initialized. Run Cell 2, then rerun Cell 14 and this cell.")

# Use the full dataset (1000 samples) for 0.5B grid runs.
data_full_05 = data
total_samples_05 = sum(len(v) for v in data_full_05.values())
breakdown_05 = {k: len(v) for k, v in data_full_05.items()}
print(f"Using full dataset for 0.5B grid: {total_samples_05} samples")
print(f"Dataset breakdown: {breakdown_05}")

for k_val in DRAFT_LENGTHS:
    for regime_name in REGIMES:
        config_key = f"0.5B_k{k_val}_{regime_name}"
        if config_key in spec_results_05 and spec_results_05[config_key]:
            print(f"Skipping existing config: {config_key}")
            continue

        print(f"{'=' * 60}")
        print(f"Speculative: draft=0.5B, k={k_val}, regime={regime_name}, samples={total_samples_05}")
        print(f"{'=' * 60}")

        results = spec_grid_runner(
            data_full_05, "0.5B", k_val, regime_name,
            target_model, target_tokenizer,
            draft_05_model, draft_05_tokenizer,
        )
        spec_results_05[config_key] = results

print(f"Grid A ready: {len(spec_results_05)} configs")


In [ ]:
# --- Grid A: Evaluate quality and compute metrics ---
from metrics import compute_acceptance_metrics, compute_speedup, compute_quality_delta, compute_latency_metrics
from evaluate import evaluate_results

grid_a_summary = []
for config_key, results in spec_results_05.items():
    parts = config_key.split("_")  # e.g. '0.5B_k4_deterministic'
    k_val = int(parts[1][1:])
    regime = parts[2]

    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    base_quality_ref = base_quality_det if regime == "deterministic" else base_quality_stoch

    lat = compute_latency_metrics(results)
    acc = compute_acceptance_metrics(results)
    quality = evaluate_results(results, data)
    speedup = compute_speedup(baseline_ref, results)
    delta_q = compute_quality_delta(base_quality_ref, quality)

    print(f"\n{config_key}: S={speedup:.2f}x, α={acc['alpha']:.3f}, B_eff={acc['B_eff']:.1f}")
    for task_k, dq in delta_q.items():
        print(f"  {task_k}: {dq:+.2f}pp")

    grid_a_summary.append({
        "config": config_key, "draft": "0.5B", "k": k_val, "regime": regime,
        **lat, **acc, "S": speedup, **quality, **delta_q,
    })

df_grid_a = pd.DataFrame(grid_a_summary)
df_grid_a

## Phase 4 — Speculative Grid B: Qwen2.5-1.5B Draft

Run speculative decoding with 1.5B draft across the same 6 configs, plus stability analysis.

In [ ]:
import os
import sys
from pathlib import Path
import torch

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import gc
from runtime import bootstrap_notebook, ensure_data, ensure_target_model, ensure_baseline_results, ensure_spec_results, ensure_draft_model
from speculative import run_speculative_grid
from config import TARGET_MODEL_ID

bootstrap_notebook()
if "install_notebook_progress_helpers" in globals():
    install_notebook_progress_helpers()

data = ensure_data(globals())
ensure_baseline_results(globals())
spec_results_05 = ensure_spec_results(globals(), "0.5B")
spec_results_15 = ensure_spec_results(globals(), "1.5B")
target_model, target_tokenizer = ensure_target_model(globals())
loaded_target = str(getattr(target_model, "name_or_path", ""))
print(f"Configured target model: {TARGET_MODEL_ID}")
print(f"Loaded target model: {loaded_target}")
assert loaded_target == TARGET_MODEL_ID, f"Target mismatch: loaded={loaded_target}, expected={TARGET_MODEL_ID}"

# Free 0.5B draft to save memory, then load 1.5B
if "draft_05_model" in globals():
    del draft_05_model
if "draft_05_tokenizer" in globals():
    del draft_05_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

draft_15_model, draft_15_tokenizer = ensure_draft_model(globals(), "1.5B")


In [ ]:
from config import DRAFT_LENGTHS, REGIMES

spec_results_15 = globals().get("spec_results_15", {})
spec_grid_runner = globals().get("notebook_run_speculative_grid")
if spec_grid_runner is None:
    raise RuntimeError("Notebook progress runner is not initialized. Run Cell 2, then rerun Cell 19 and this cell.")

# Use the full dataset (1000 samples) for 1.5B grid runs.
data_full_15 = data
total_samples_15 = sum(len(v) for v in data_full_15.values())
breakdown_15 = {k: len(v) for k, v in data_full_15.items()}
print(f"Using full dataset for 1.5B grid: {total_samples_15} samples")
print(f"Dataset breakdown: {breakdown_15}")

for k_val in DRAFT_LENGTHS:
    for regime_name in REGIMES:
        config_key = f"1.5B_k{k_val}_{regime_name}"
        if config_key in spec_results_15 and spec_results_15[config_key]:
            print(f"Skipping existing config: {config_key}")
            continue

        print(f"{'=' * 60}")
        print(f"Speculative: draft=1.5B, k={k_val}, regime={regime_name}, samples={total_samples_15}")
        print(f"{'=' * 60}")

        results = spec_grid_runner(
            data_full_15, "1.5B", k_val, regime_name,
            target_model, target_tokenizer,
            draft_15_model, draft_15_tokenizer,
        )
        spec_results_15[config_key] = results

print(f"Grid B ready: {len(spec_results_15)} configs")


In [ ]:
# --- Grid B: Evaluate quality and compute metrics ---
grid_b_summary = []
for config_key, results in spec_results_15.items():
    parts = config_key.split("_")
    k_val = int(parts[1][1:])
    regime = parts[2]

    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    base_quality_ref = base_quality_det if regime == "deterministic" else base_quality_stoch

    lat = compute_latency_metrics(results)
    acc = compute_acceptance_metrics(results)
    quality = evaluate_results(results, data)
    speedup = compute_speedup(baseline_ref, results)
    delta_q = compute_quality_delta(base_quality_ref, quality)

    print(f"\n{config_key}: S={speedup:.2f}x, α={acc['alpha']:.3f}, B_eff={acc['B_eff']:.1f}")
    for task_k, dq in delta_q.items():
        print(f"  {task_k}: {dq:+.2f}pp")

    grid_b_summary.append({
        "config": config_key, "draft": "1.5B", "k": k_val, "regime": regime,
        **lat, **acc, "S": speedup, **quality, **delta_q,
    })

df_grid_b = pd.DataFrame(grid_b_summary)
df_grid_b

In [ ]:
# --- Combine all 12 configs into master table ---
df_all = pd.concat([df_grid_a, df_grid_b], ignore_index=True)
print("\nFull 12-config results matrix:")
display(df_all[["config", "draft", "k", "regime", "S", "alpha", "B_eff",
                "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms"]])

# Save master table
df_all.to_csv(RESULTS_DIR / "all_configs_summary.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'all_configs_summary.csv'}")

### Phase 4b — Stability Analysis (Top 2 Configs)

Identify the top-2 configs by speedup (subject to |ΔQ| ≤ 1.0) and re-run with seeds {42, 123, 999}.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from runtime import bootstrap_notebook, ensure_data, ensure_baseline_results, ensure_spec_results, ensure_df_all
from metrics import compute_speedup_stability
from speculative import run_stability_analysis

bootstrap_notebook()

data = ensure_data(globals())
ensure_baseline_results(globals())
spec_results_05 = ensure_spec_results(globals(), "0.5B")
spec_results_15 = ensure_spec_results(globals(), "1.5B")
if "df_all" not in globals() or df_all.empty:
    df_all = ensure_df_all(globals())


In [ ]:
# Run stability analysis for top 2 configs
stability_results = {}

for _, row in top2.iterrows():
    draft_label = row["draft"]
    k_val = int(row["k"])
    regime = row["regime"]
    config_key = row["config"]

    print(f"\n{'=' * 60}")
    print(f"Stability analysis: {config_key}")
    print(f"{'=' * 60}")

    # Load correct draft model
    if draft_label == "0.5B":
        dm, dt = load_draft_model("0.5B")
    else:
        dm, dt = draft_15_model, draft_15_tokenizer

    seed_runs = run_stability_analysis(
        data, draft_label, k_val, regime,
        target_model, target_tokenizer, dm, dt,
    )

    # Compute speedup per seed
    baseline_ref = baseline_det if regime == "deterministic" else baseline_stoch
    speedups = []
    for sr in seed_runs:
        s = compute_speedup(baseline_ref, sr["results"])
        speedups.append(s)
        print(f"  seed={sr['seed']}: S={s:.4f}")

    sigma = compute_speedup_stability(speedups)
    print(f"  σ_S = {sigma:.4f}")
    stability_results[config_key] = {"speedups": speedups, "sigma_S": sigma}

print("\n✓ Stability analysis complete")

## Phase 5 — Quality and Error Analysis

Systematic analysis of disagreement patterns between speculative and baseline outputs, including length buckets, taxonomy, and rejection behavior proxies.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from difflib import SequenceMatcher
import numpy as np
from evaluate import cnndm_rouge_l
from runtime import bootstrap_notebook, ensure_data, ensure_baseline_results, ensure_spec_results

bootstrap_notebook()

data = ensure_data(globals())
ensure_baseline_results(globals())
spec_results_05 = ensure_spec_results(globals(), "0.5B")
spec_results_15 = ensure_spec_results(globals(), "1.5B")

# Merge all speculative runs
all_spec_results = {**spec_results_05, **spec_results_15}

# Baseline lookup by regime + sample_id
baseline_lookup = {
    "deterministic": {r["sample_id"]: r for r in baseline_det},
    "stochastic": {r["sample_id"]: r for r in baseline_stoch},
}

rows = []
for config_name, cfg_rows in all_spec_results.items():
    regime = cfg_rows[0]["regime"] if cfg_rows else "deterministic"
    for r in cfg_rows:
        sid = r["sample_id"]
        base = baseline_lookup[regime].get(sid)
        if base is None:
            continue

        pred_text = (r.get("output_text") or "").strip()
        base_text = (base.get("output_text") or "").strip()

        pred_words = len(pred_text.split())
        base_words = len(base_text.split())
        disagree = pred_text != base_text
        sim_ratio = SequenceMatcher(None, pred_text, base_text).ratio()

        if pred_words <= 16:
            length_bucket = "short"
        elif pred_words <= 64:
            length_bucket = "medium"
        else:
            length_bucket = "long"

        rouge_to_base = np.nan
        if r["task"] == "cnndm" and pred_text and base_text:
            rouge_to_base = cnndm_rouge_l(pred_text, base_text)

        rows.append({
            "config": config_name,
            "draft": r["draft"],
            "k": r["k"],
            "regime": regime,
            "task": r["task"],
            "sample_id": sid,
            "disagree": disagree,
            "pred_words": pred_words,
            "base_words": base_words,
            "length_bucket": length_bucket,
            "sim_ratio": sim_ratio,
            "rouge_to_base": rouge_to_base,
            "alpha_sample": (r.get("total_accepted", 0) / r.get("total_proposed", 1)) if r.get("total_proposed", 0) > 0 else np.nan,
        })

df_phase5 = pd.DataFrame(rows)
print(f"Constructed Phase 5 analysis table: {len(df_phase5)} rows")
display(df_phase5.head(10))


In [ ]:
# Disagreement rates by config/task and by output length bucket

df_disagree = df_phase5[df_phase5["disagree"]].copy()

disagree_rate = (
    df_phase5.groupby(["config", "task"], as_index=False)["disagree"]
    .mean()
    .rename(columns={"disagree": "disagreement_rate"})
)

bucket_dist = (
    df_disagree.groupby(["config", "length_bucket"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
if not bucket_dist.empty:
    bucket_dist["pct_within_config"] = (
        bucket_dist["count"]
        / bucket_dist.groupby("config")["count"].transform("sum")
        * 100
    ).round(2)

print("Disagreement rate by config/task:")
display(disagree_rate.sort_values(["disagreement_rate", "config"], ascending=[False, True]).head(20))

print("\nLength bucket distribution among disagreement cases:")
display(bucket_dist.sort_values(["config", "length_bucket"]))

In [ ]:
# Error taxonomy + rejection clustering proxy

def classify_error(row):
    if not row["disagree"]:
        return "match"

    pred_w = max(row["pred_words"], 1)
    base_w = max(row["base_words"], 1)

    # Truncation: output substantially shorter than baseline output.
    if pred_w < 0.6 * base_w:
        return "truncation"

    # Semantic drift: low lexical similarity and (for CNNDM) low ROUGE-L to baseline output.
    if row["sim_ratio"] < 0.25:
        return "semantic_drift"
    if row["task"] == "cnndm" and not np.isnan(row["rouge_to_base"]) and row["rouge_to_base"] < 0.2:
        return "semantic_drift"

    return "substitution"


df_phase5["error_type"] = df_phase5.apply(classify_error, axis=1)

taxonomy = (
    df_phase5[df_phase5["error_type"] != "match"]
    .groupby(["config", "error_type"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
if not taxonomy.empty:
    taxonomy["pct_within_config"] = (
        taxonomy["count"]
        / taxonomy.groupby("config")["count"].transform("sum")
        * 100
    ).round(2)

# Rejection-position clustering requires verify-step position logs.
# Current CSV rows do not keep verify_log, so we provide an acceptance-ratio proxy.
proxy = df_phase5.dropna(subset=["alpha_sample"]).copy()
proxy["rejection_proxy"] = pd.cut(
    proxy["alpha_sample"],
    bins=[-1, 0.33, 0.66, 1.0],
    labels=["early_reject_proxy", "mid_reject_proxy", "late_reject_proxy"],
)
rej_proxy_table = (
    proxy.groupby(["config", "rejection_proxy"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)
if not rej_proxy_table.empty:
    rej_proxy_table["pct_within_config"] = (
        rej_proxy_table["count"]
        / rej_proxy_table.groupby("config")["count"].transform("sum")
        * 100
    ).round(2)

has_verify_logs = any("verify_log" in row for cfg in all_spec_results.values() for row in cfg)
print(f"verify_log persisted in run outputs: {has_verify_logs}")
if not has_verify_logs:
    print("NOTE: true token-position rejection clustering is not possible from current saved rows; using acceptance-ratio proxy instead.")

print("\nError taxonomy among disagreement cases:")
display(taxonomy.sort_values(["config", "count"], ascending=[True, False]))

print("\nRejection clustering proxy:")
display(rej_proxy_table.sort_values(["config", "rejection_proxy"]))

## Phase 6 — Synthesis and Visualization

Generate Pareto/acceptance figures, synthesis tables, and an execution-plan audit for missing items.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
project_root = None
for c in candidates:
    if (c / "src").exists() and (c / "experiment.ipynb").exists():
        project_root = c
        break
if project_root is None:
    project_root = cwd

SRC_DIR = str(project_root / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

os.environ["SPECDEC_HF_OFFLINE_FIRST"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import matplotlib.pyplot as plt
import seaborn as sns
from runtime import bootstrap_notebook, ensure_df_all

bootstrap_notebook()

if "df_all" not in globals() or df_all.empty:
    df_all = ensure_df_all(globals())
if "stability_results" not in globals():
    stability_results = {}

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

df_plot = df_all.copy()
delta_cols = [c for c in df_plot.columns if c.startswith("delta_")]
if not delta_cols:
    delta_cols = [c for c in df_plot.columns if c.startswith("dQ_")]

df_plot["max_abs_delta"] = df_plot[delta_cols].abs().max(axis=1) if delta_cols else 0.0

# Pareto scatter: Speedup vs max quality drift
plt.figure(figsize=(8, 5))
ax = sns.scatterplot(
    data=df_plot,
    x="S",
    y="max_abs_delta",
    hue="draft",
    style="regime",
    s=90,
)
ax.set_title("Pareto: Speedup vs Max |Delta Q|")
ax.set_xlabel("Speedup S (x)")
ax.set_ylabel("Max |Delta Q| (pp)")
plt.tight_layout()
pareto_path = FIGURES_DIR / "pareto_speedup_vs_quality_delta.png"
plt.savefig(pareto_path, dpi=160)
plt.show()

# Acceptance vs k grouped by draft and regime
alpha_k = (
    df_plot.groupby(["draft", "regime", "k"], as_index=False)["alpha"]
    .mean()
)

plt.figure(figsize=(8, 5))
ax = sns.lineplot(data=alpha_k, x="k", y="alpha", hue="draft", style="regime", marker="o")
ax.set_title("Acceptance Rate alpha vs Draft Length k")
ax.set_xlabel("k")
ax.set_ylabel("alpha")
plt.tight_layout()
alpha_k_path = FIGURES_DIR / "acceptance_vs_k.png"
plt.savefig(alpha_k_path, dpi=160)
plt.show()

# Acceptance by regime (temperature proxy)
alpha_reg = (
    df_plot.groupby(["regime", "draft"], as_index=False)["alpha"]
    .mean()
)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=alpha_reg, x="regime", y="alpha", hue="draft")
ax.set_title("Acceptance Rate alpha by Regime")
ax.set_xlabel("Regime")
ax.set_ylabel("alpha")
plt.tight_layout()
alpha_reg_path = FIGURES_DIR / "acceptance_by_regime.png"
plt.savefig(alpha_reg_path, dpi=160)
plt.show()

print("Saved figures:")
print(f"  - {pareto_path}")
print(f"  - {alpha_k_path}")
print(f"  - {alpha_reg_path}")


In [ ]:
# Synthesis tables for manuscript

results_matrix_cols = [
    "config", "draft", "k", "regime", "S", "alpha", "B_eff",
    "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms",
    "gsm8k", "mmlu", "cnndm",
]
results_matrix_cols = [c for c in results_matrix_cols if c in df_all.columns]

results_matrix = df_all[results_matrix_cols].sort_values(["draft", "k", "regime"]).copy()
display(results_matrix)
results_matrix.to_csv(RESULTS_DIR / "table_full_results_matrix.csv", index=False)

# Best speculative config per task + baseline comparison
quality_table_rows = []
baseline_quality_map = {
    "deterministic": base_quality_det,
    "stochastic": base_quality_stoch,
}

for task in ["gsm8k", "mmlu", "cnndm"]:
    if task not in df_all.columns:
        continue
    best_idx = df_all[task].idxmax()
    best_row = df_all.loc[best_idx]
    regime = best_row["regime"]
    baseline_score = baseline_quality_map[regime][task]
    quality_table_rows.append({
        "task": task,
        "baseline_regime": regime,
        "baseline_score": baseline_score,
        "best_spec_config": best_row["config"],
        "best_spec_score": best_row[task],
        "delta_spec_minus_base": round(best_row[task] - baseline_score, 2),
    })

df_quality_compare = pd.DataFrame(quality_table_rows)
print("Best speculative config per task (vs matching baseline regime):")
display(df_quality_compare)
df_quality_compare.to_csv(RESULTS_DIR / "table_quality_comparison.csv", index=False)

print("Saved tables:")
print(f"  - {RESULTS_DIR / 'table_full_results_matrix.csv'}")
print(f"  - {RESULTS_DIR / 'table_quality_comparison.csv'}")

In [ ]:
# Audit checklist: what is still missing for Phase 6 paper completion
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "sec").exists():
    # Fallback to notebook path root behavior if executed from elsewhere
    project_root = Path(SRC_DIR).parent

checks = []

# 1) Missing sections requested by execution plan
for rel in ["sec/01b_related_work.tex", "sec/06_results.tex"]:
    p = project_root / rel
    checks.append({
        "item": f"Section file exists: {rel}",
        "status": "PASS" if p.exists() else "MISSING",
        "evidence": str(p),
    })

# 2) Abstract placeholders still present?
abstract_path = project_root / "sec/00_abstract.tex"
if abstract_path.exists():
    abstract_text = abstract_path.read_text(encoding="utf-8", errors="ignore")
    placeholders_present = any(tok in abstract_text for tok in ["[X]", "[Y]", "[Z]"])
    checks.append({
        "item": "Abstract placeholders resolved ([X],[Y],[Z])",
        "status": "MISSING" if placeholders_present else "PASS",
        "evidence": "Placeholders found" if placeholders_present else "No placeholders found",
    })

# 3) Rejection-position instrumentation availability
has_verify_logs = any("verify_log" in row for cfg in all_spec_results.values() for row in cfg)
checks.append({
    "item": "Token-position rejection logs persisted",
    "status": "PASS" if has_verify_logs else "MISSING",
    "evidence": "verify_log present in result rows" if has_verify_logs else "verify_log dropped in run_speculative_grid output rows",
})

# 4) Visualization module implementation
viz_path = project_root / "src/visualize.py"
if viz_path.exists():
    viz_text = viz_path.read_text(encoding="utf-8", errors="ignore")
    stub = "implemented later" in viz_text.lower()
    checks.append({
        "item": "src/visualize.py implemented",
        "status": "MISSING" if stub else "PASS",
        "evidence": "Stub marker found" if stub else "No stub marker found",
    })

# 5) Ensure phase outputs written
required_outputs = [
    project_root / "results/all_configs_summary.csv",
    project_root / "results/all_configs_combined.csv",
    project_root / "results/table_full_results_matrix.csv",
    project_root / "results/table_quality_comparison.csv",
]
for out in required_outputs:
    checks.append({
        "item": f"Output generated: {out.name}",
        "status": "PASS" if out.exists() else "PENDING",
        "evidence": str(out),
    })

df_audit = pd.DataFrame(checks)
print("Phase 6 audit report:")
display(df_audit)

missing_items = df_audit[df_audit["status"].isin(["MISSING", "PENDING"])]
if missing_items.empty:
    print("\nAll checklist items passed.")
else:
    print("\nItems still pending/missing:")
    display(missing_items)

## Phase 7 — ACSD: Adaptive Cascaded Speculative Decoding

Phases 2–6 establish the autoregressive speculative baseline. Phase 7 replaces
the prospective DriftDiffuse path with **Adaptive Cascaded Speculative Decoding
(ACSD)**, designed for practical speed on a consumer GPU with stable quality.

ACSD keeps standard target-side verification but adds two online controls:

1. **Adaptive draft length**: choose $k\in\{4,8,16\}$ from rolling acceptance.
2. **Cascaded drafter switch**: use 0.5B as primary, switch to 1.5B rescue when
   acceptance collapses, then return to 0.5B when stable.
3. **Tail guardrail**: if acceptance remains very low after enough generated
   tokens, fall back to target-only AR for the remainder of that sample.

This directly targets wall-clock tails that made fixed-policy DriftDiffuse runs
slow in practice.

Primary outputs in this phase:

- ACSD per-sample CSVs by regime
- ACSD aggregate summary (speed, acceptance, quality deltas)
- Head-to-head comparison against best AR speculative configs

In [ ]:
# --- Phase 7: ACSD setup (target + primary/rescue drafts) ---
import os
import sys
import gc
from pathlib import Path

import torch

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_root = next((p for p in candidates if (p / "src").exists()), None)
assert project_root is not None, "Could not locate project root containing src/"
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from config import ACSD_EVAL, TARGET_MODEL_ID
from runtime import bootstrap_notebook, ensure_data, ensure_target_model, ensure_draft_model

bootstrap_notebook()

for name in ("drifter_model", "drifter_schedule", "draft_15_model", "draft_15_tokenizer", "draft_05_model", "draft_05_tokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

data = ensure_data(globals())
target_model, target_tokenizer = ensure_target_model(globals())

primary_label = ACSD_EVAL["primary_draft"]
rescue_label = ACSD_EVAL["rescue_draft"]
draft_primary_model, _ = ensure_draft_model(globals(), primary_label)
draft_rescue_model, _ = ensure_draft_model(globals(), rescue_label)

print(f"Target model: {TARGET_MODEL_ID}")
print(f"Primary draft: {primary_label}")
print(f"Rescue draft:  {rescue_label}")
print(f"ACSD k choices: {ACSD_EVAL['k_choices']} (base={ACSD_EVAL['base_k']})")
print(f"ACSD regimes: {ACSD_EVAL['regimes']}")

Loading target model: Qwen/Qwen2.5-3B-Instruct (quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Using existing drifter checkpoint: C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\drifter_ckpt\drifter_latest.pt
DriftDiffuser loaded — 96.9M params
  hidden=512, layers=6, heads=8, k_max=16, max_ctx_len=768
  vocab_size=151665 (matches target tokenizer: True)


In [ ]:
# --- Phase 7: run ACSD over full evaluation manifests ---
import gc
import importlib
import time
from datetime import datetime, timedelta

import torch
from tqdm.auto import tqdm

from config import ACSD_EVAL
import acsd as acsd_module

acsd_module = importlib.reload(acsd_module)
run_acsd_grid = acsd_module.run_acsd_grid

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

draft_models = {
    ACSD_EVAL["primary_draft"]: draft_primary_model,
    ACSD_EVAL["rescue_draft"]: draft_rescue_model,
}

regimes = tuple(ACSD_EVAL["regimes"])
samples_per_cfg = sum(len(v) for v in data.values())
total_cfg = len(regimes)
total_samples_all = samples_per_cfg * total_cfg
done_samples = [0]
started_at = time.perf_counter()

bar = tqdm(
    total=total_samples_all,
    desc="acsd samples",
    dynamic_ncols=True,
    leave=True,
    mininterval=0.2,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
)

def _eta_text(done_n: int, total_n: int) -> tuple[float, str]:
    if done_n <= 0 or done_n >= total_n:
        return 0.0, "--:--:--"
    elapsed_s = time.perf_counter() - started_at
    remaining_s = max((elapsed_s / done_n) * (total_n - done_n), 0.0)
    eta_clock = (datetime.now() + timedelta(seconds=remaining_s)).strftime("%H:%M:%S")
    return remaining_s, eta_clock

acsd_results = globals().get("acsd_results", {})

try:
    for cfg_idx, regime in enumerate(regimes, start=1):
        regime_short = "det" if regime == "deterministic" else "stoch"
        cfg_started_at = time.perf_counter()

        def _on_sample(event):
            prev = done_samples[0]
            now = prev + 1
            done_samples[0] = now
            bar.update(1)
            remaining_s, eta_clock = _eta_text(done_samples[0], total_samples_all)
            bar.set_postfix_str(
                f"cfg {cfg_idx}/{total_cfg} regime={regime_short} | sample {done_samples[0]}/{total_samples_all} | rem={remaining_s/60:.1f}m | eta={eta_clock}"
            )

        rows = run_acsd_grid(
            data=data,
            regime_name=regime,
            target_model=target_model,
            target_tokenizer=target_tokenizer,
            draft_models=draft_models,
            primary_label=ACSD_EVAL["primary_draft"],
            rescue_label=ACSD_EVAL["rescue_draft"],
            base_k=ACSD_EVAL["base_k"],
            k_choices=tuple(ACSD_EVAL["k_choices"]),
            label="acsd",
            checkpoint_every=ACSD_EVAL.get("checkpoint_every", 50),
            progress_callback=_on_sample,
            accept_window=ACSD_EVAL["accept_window"],
            k_low_threshold=ACSD_EVAL["k_low_threshold"],
            k_high_threshold=ACSD_EVAL["k_high_threshold"],
            rescue_trigger_alpha=ACSD_EVAL["rescue_trigger_alpha"],
            rescue_trigger_consecutive=ACSD_EVAL["rescue_trigger_consecutive"],
            rescue_hold_steps=ACSD_EVAL["rescue_hold_steps"],
            rescue_cooldown_steps=ACSD_EVAL.get("rescue_cooldown_steps", 8),
            ar_fallback_alpha=ACSD_EVAL["ar_fallback_alpha"],
            ar_fallback_min_tokens=ACSD_EVAL["ar_fallback_min_tokens"],
            ar_fallback_consecutive=ACSD_EVAL.get("ar_fallback_consecutive", 3),
        )

        key = f"acsd_{ACSD_EVAL['primary_draft']}_to_{ACSD_EVAL['rescue_draft']}_{regime}"
        acsd_results[key] = rows

        cfg_elapsed = time.perf_counter() - cfg_started_at
        remaining_s, eta_clock = _eta_text(done_samples[0], total_samples_all)
        bar.set_postfix_str(
            f"cfg {cfg_idx}/{total_cfg} done regime={regime_short} | cfg={cfg_elapsed:.1f}s | sample {done_samples[0]}/{total_samples_all} | rem={remaining_s/60:.1f}m | eta={eta_clock}"
        )
finally:
    bar.close()

globals()["acsd_results"] = acsd_results

print(f"ACSD configs processed: {len(acsd_results)}")
for key, rows in acsd_results.items():
    print(f"  {key}: {len(rows)} samples")

  [gsm8k] loaded 300 samples from manifest
  [mmlu] loaded 500 samples from manifest
  [cnndm] loaded 200 samples from manifest


drift samples:   0%|          | 0/4000 [00:00<?, ?it/s]

Saved -> C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\drift_drift_n2_block_k8_det.csv
Saved -> C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\drift_drift_n3_block_k8_det.csv
Saved -> C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\drift_drift_n2_block_k16_det.csv
Saved -> C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\drift_drift_n3_block_k16_det.csv
DriftDiffuse configs processed: 4
  drift_n2_block_k8_deterministic: 1000 samples
  drift_n3_block_k8_deterministic: 1000 samples
  drift_n2_block_k16_deterministic: 1000 samples
  drift_n3_block_k16_deterministic: 1000 samples


In [ ]:
# --- Phase 8: ACSD metrics + head-to-head comparison vs AR best ---
import pandas as pd

from metrics import (
    compute_acceptance_metrics,
    compute_latency_metrics,
    compute_speedup,
    compute_quality_delta,
)
from evaluate import evaluate_results
from config import RESULTS_DIR, ACSD_EVAL

baseline_results = globals().get("baseline_results", {})
baseline_det_rows = baseline_results.get("deterministic", [])
if not baseline_det_rows:
    baseline_det_rows = globals().get("baseline_det", [])

data = globals().get("data")
if data is None:
    from runtime import ensure_data
    data = ensure_data(globals())

acsd_summary = []
for key, rows in acsd_results.items():
    regime = key.split("_")[-1]
    accept = compute_acceptance_metrics(rows)
    latency = compute_latency_metrics(rows)
    quality = evaluate_results(rows, data)

    if baseline_det_rows:
        base_qual = evaluate_results(baseline_det_rows, data)
        speedup = compute_speedup(baseline_det_rows, rows)
        delta_q = compute_quality_delta(base_qual, quality)
    else:
        speedup = 0.0
        delta_q = {}

    df_rows = pd.DataFrame(rows)
    rescue_steps_mean = float(df_rows["acsd_rescue_steps"].mean()) if "acsd_rescue_steps" in df_rows.columns else 0.0
    fallback_rate = float(df_rows["acsd_used_ar_fallback"].mean()) if "acsd_used_ar_fallback" in df_rows.columns else 0.0

    acsd_summary.append({
        "config": f"ACSD-{regime}",
        "draft": "ACSD",
        "regime": regime,
        "primary_draft": ACSD_EVAL["primary_draft"],
        "rescue_draft": ACSD_EVAL["rescue_draft"],
        "base_k": ACSD_EVAL["base_k"],
        "alpha": accept.get("alpha"),
        "B_eff": accept.get("B_eff"),
        "tokens_per_sec": latency.get("R_tok_mean"),
        "speedup": speedup,
        "mean_rescue_steps": round(rescue_steps_mean, 3),
        "fallback_rate": round(fallback_rate, 4),
        **{f"quality_{k_}": v for k_, v in quality.items()},
        **{f"delta_{k_}": v for k_, v in delta_q.items()},
    })

df_acsd = pd.DataFrame(acsd_summary).sort_values(["regime"]).reset_index(drop=True)
print("ACSD summary:")
display(df_acsd)

df_all = globals().get("df_all")
if df_all is not None and not df_all.empty:
    ar_det = df_all[df_all["regime"] == "deterministic"].copy()
    ar_best = ar_det.sort_values("S", ascending=False).head(3)
    acsd_best = df_acsd.sort_values("speedup", ascending=False).head(3)
    print("\nTop-3 AR (deterministic):")
    display(ar_best[["config", "draft", "k", "alpha", "B_eff", "R_tok_mean", "S"]])
    print("\nTop ACSD configs:")
    display(acsd_best[["config", "regime", "alpha", "B_eff", "tokens_per_sec", "speedup", "mean_rescue_steps", "fallback_rate"]])

df_acsd.to_csv(RESULTS_DIR / "acsd_summary.csv", index=False)
print(f"\nSaved -> {RESULTS_DIR / 'acsd_summary.csv'}")

  gsm8k (exact_match): 0.00%  (n=300)
  mmlu (letter_match): 25.80%  (n=500)
  cnndm (rouge_l): 2.73%  (n=200)
  gsm8k (exact_match): 0.00%  (n=300)
  mmlu (letter_match): 25.80%  (n=500)
  cnndm (rouge_l): 2.74%  (n=200)
  gsm8k (exact_match): 0.00%  (n=300)
  mmlu (letter_match): 25.80%  (n=500)
  cnndm (rouge_l): 1.81%  (n=200)
  gsm8k (exact_match): 0.00%  (n=300)
  mmlu (letter_match): 25.80%  (n=500)
  cnndm (rouge_l): 1.80%  (n=200)
DriftDiffuse summary:


,config,draft,k,regime,n_denoise,accept_mode,alpha,B_eff,tokens_per_sec,speedup,quality_gsm8k,quality_mmlu,quality_cnndm
0,DriftDiffuse-n2-block,DriftDiffuse,8,deterministic,2,block,2141.8522,16838.30,66.95,0.0,0.0,25.8,2.73
1,DriftDiffuse-n3-block,DriftDiffuse,8,deterministic,3,block,2131.3053,16757.84,56.79,0.0,0.0,25.8,2.74
2,DriftDiffuse-n2-block,DriftDiffuse,16,deterministic,2,block,1340.3763,20509.66,142.53,0.0,0.0,25.8,1.81
3,DriftDiffuse-n3-block,DriftDiffuse,16,deterministic,3,block,1338.8079,20484.08,98.50,0.0,0.0,25.8,1.80



Saved -> C:\Working\speculative-decoding-main_v8\speculative-decoding-main\results\drift_summary.csv


### Transition: Phase 7 → Phase 8

Phase 7 executes ACSD decoding runs. Phase 8 consolidates ACSD metrics and
compares them against the AR speculative baseline under the same manifests and
evaluation protocol.

## Summary

Display the final results matrix for baseline + AR speculative runs, together
with ACSD Phase 7/8 outcomes and stability info from Phase 4b.

In [6]:
# Final summary table
print("=" * 80)
print("EXPERIMENT COMPLETE — FULL RESULTS MATRIX")
print("=" * 80)

display_cols = [c for c in ["config", "draft", "k", "regime", "S", "alpha", "B_eff",
                "T_mean_s", "R_tok_mean", "TTFT_mean_ms", "TPOT_mean_ms"] if c in df_all.columns]

print("\n--- Latency & Throughput ---")
display(df_all[display_cols].sort_values("R_tok_mean", ascending=False))

quality_cols = [c for c in df_all.columns if c in ["gsm8k", "mmlu", "cnndm"]]
if quality_cols:
    print("\n--- Quality (%) ---")
    display(df_all[["config"] + quality_cols])

# Best config under quality constraint
if not df_qualified.empty:
    best = df_qualified.loc[df_qualified["S"].idxmax()]
    print(f"\n★ Best config (|ΔQ| ≤ 1.0): {best['config']}")
    print(f"  Speedup: {best['S']:.2f}x, α: {best['alpha']:.3f}")

# Save combined table
df_all.to_csv(RESULTS_DIR / "all_configs_combined.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'all_configs_combined.csv'}")

# Stability
if stability_results:
    print("\n--- Stability (σ_S) ---")
    for cfg, info in stability_results.items():
        print(f"  {cfg}: σ_S = {info['sigma_S']:.4f}, seeds = {info['speedups']}")

EXPERIMENT COMPLETE — FULL RESULTS MATRIX


AttributeError: 'NoneType' object has no attribute 'columns'